In [ ]:
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from datasets import ImageDataset
from distortions import *
from models import DistortionBinaryClassifier, IQAEncoder
from torchvision import transforms
from torch.utils.data import DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(model_path, device):
    model = IQAEncoder(feature_dim=128, model_name='resnet50').to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

encoder = load_model('models/resnet50_128_out.pth', device)
model = DistortionBinaryClassifier(encoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

distortions = [Clean(), LensBlur(), MotionBlur(), GaussianNoise(), Overexposure(), Underexposure(), Compression(), Ghosting(), Aliasing()]
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)), # 224 or 384
    transforms.ToTensor(),
])
image_folder = "data/video_frames_4" # "data/FLIR_ADAS_v2/images_thermal_train/data"
image_paths = [os.path.join(image_folder, fname) for fname in os.listdir(image_folder) if fname.endswith(('.jpg', '.png'))]
dataset = ImageDataset(image_paths, distortions=distortions, transform=transform, binary_labels=True)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4)
print(f"Train Dataset length: {len(dataset)}")

image_folder = "data/video_frames_5" # "data/FLIR_ADAS_v2/images_thermal_val/data"
image_paths = [os.path.join(image_folder, fname) for fname in os.listdir(image_folder) if fname.endswith(('.jpg', '.png'))]
eval_dataset = ImageDataset(image_paths, distortions=distortions, transform=transform, binary_labels=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=4)
print(f"Eval Dataset length: {len(eval_dataset)}")

epochs = 100
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for imgs, labels in tqdm(dataloader, desc=f"Binary Epoch {epoch+1}", leave=False):
        imgs = imgs.to(device)
        binary_labels = torch.tensor([0.0 if l == 'Clean' else 1.0 for l in labels], dtype=torch.float32, device=device).unsqueeze(1)  # shape: [B, 1]

        logits = model(imgs)
        loss = loss_fn(logits, binary_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss / len(dataloader):.4f}")

    # Validation
    if ((epoch+1) % 10 == 0) or (epoch == 0):
        accuracy = 0.0
        model.eval()
        with torch.no_grad():
            for imgs, labels in tqdm(eval_dataloader, desc=f"Eval Epoch {epoch+1}", leave=False):
                imgs = imgs.to(device)
                binary_labels = torch.tensor([0.0 if l == 'Clean' else 1.0 for l in labels], dtype=torch.float32, device=device).unsqueeze(1)
                logits = model(imgs)
                preds = torch.sigmoid(logits) > 0.5
                accuracy += (preds == binary_labels).float().mean().item()

        accuracy /= len(eval_dataloader)
        print(f"Eval Accuracy: {accuracy:.4f}")


Train Dataset length: 3035
Eval Dataset length: 4045


Epoch 1, Loss: 0.2199


Eval Accuracy: 0.8566
Eval Quality: 0.4674


Epoch 2, Loss: 0.1046


Epoch 3, Loss: 0.0854


Epoch 4, Loss: 0.0688


KeyboardInterrupt: 